The goal of this notebook is to investigate and quantify the change in chlorophyll over the lifetimes of cyclones traveling South and anticyclones traveling North respectively.

What is Known:
- Cyclones tend to have enhanced chlorophyll while anticyclones have suppressed chlorophyl.
- The north side of the Gulf Stream contains cooler nutrient-rich waters while south of the Gulf Stream contains warmer and more oligotrophic waters.

Target eddies: a cyclone that was born north of the daily Gulf Stream axis and ended south, or was born within `NEAR_AXIS_KM` (150 km) of the axis and ended south. An anticyclone uses the mirror rule: born south and ended north, or born within `NEAR_AXIS_KM` of the axis and ended north. The birth and death sides and the signed axis distances come from `silver/gulf_stream/eddy_movement.parquet`, written by the `gulf_stream` pipeline stage.

Chlorophyll source: the `CHL` field of the Copernicus Marine daily L3 plankton product (multi-sensor GlobColour processing, 4 km), composited over NASA 8-day periods, in mg/m³. Run the `download_cmems` and `collocate_plankton` pipeline stages to create `gold/eddy_plankton_table.parquet` before this notebook. That stage applies the thresholds in the experiment configuration and covers the whole tracking window, so every eddy has chlorophyll from its first detection. Each eddy uses its observed boundary nearest the composite midpoint. The mean uses finite interior pixels, with at least 50% coverage and 10 valid pixels. This calculation does not use PACE or SDP. Within each age bin, the composites of one eddy are averaged into a single value first, so an eddy counts once in the bin no matter how many composites it has there. The bin value is the plain mean of those per-eddy values.


In [ ]:
# Count independent tracks in all four endpoint classes; mark the southbound-cyclone and northbound-anticyclone target eddies, and print the number with CHL observations.
from pathlib import Path
from typing import cast
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.axes import Axes

PROJECT_ROOT = Path('/Users/jerry/school/research/eddy-tracking')
sys.path.insert(0, str(PROJECT_ROOT))
from utils.config import load_config

EXPERIMENT = 'gulf_stream_20240305_20260531'
N_AGE_BINS = 5
N_BOOTSTRAP = 2000
RANDOM_SEED = 2026
EXCLUDE_RECORD_EDGE_TRACKS = False
NEAR_AXIS_KM = 150
DATA_DIR = PROJECT_ROOT / 'data' / EXPERIMENT
cfg = load_config(EXPERIMENT)
polarity_names = ('cyclone', 'anticyclone')
target_classes = {'cyclone': 'NS', 'anticyclone': 'SN'}
polarity_colors = {'cyclone': '#2166ac', 'anticyclone': '#b2182b'}
target_labels = {'cyclone': 'Cyclones ending south of the axis', 'anticyclone': 'Anticyclones ending north of the axis'}
identity_columns = ['polarity', 'track_id']

movement = pd.read_parquet(DATA_DIR / 'silver/gulf_stream/eddy_movement.parquet')
chl = pd.read_parquet(DATA_DIR / 'gold/eddy_plankton_table.parquet')
eddy_tracks = movement.merge(
    cast(pd.Series, chl.groupby(identity_columns).size()).rename('n_chl_dates').reset_index(),
    on=identity_columns, how='left',
)
eddy_tracks['n_chl_dates'] = eddy_tracks['n_chl_dates'].fillna(0).astype(int)
physical_start, physical_end = pd.to_datetime(cfg['base']['time']['eddy_date_range'])
eddy_tracks['at_record_edge'] = (
    (eddy_tracks['birth_date'] <= physical_start)
    | (eddy_tracks['death_date'] >= physical_end)
)
target_class = cast(pd.Series, eddy_tracks['polarity']).map(target_classes)
eddy_tracks['crossed_axis'] = eddy_tracks['movement'].eq(target_class)
eddy_tracks['near_axis_birth'] = (
    eddy_tracks['birth_distance_km'].abs().le(NEAR_AXIS_KM)
    & eddy_tracks['death_side'].eq(target_class.str[1])
)
eddy_tracks['is_target'] = eddy_tracks['crossed_axis'] | eddy_tracks['near_axis_birth']
chl = chl.merge(
    eddy_tracks[identity_columns + ['at_record_edge', 'is_target']],
    on=identity_columns, how='left',
)
target_chl = cast(pd.DataFrame, chl.loc[chl['is_target']]).copy()
if EXCLUDE_RECORD_EDGE_TRACKS:
    target_chl = cast(pd.DataFrame, target_chl.loc[~target_chl['at_record_edge']]).copy()

plt.rcParams.update({
    'font.size': 11, 'axes.spines.top': False, 'axes.spines.right': False,
})


classes = ['NN', 'NS', 'SN', 'SS']
count_rows = []
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), sharey=True, layout='constrained')
for panel, polarity in enumerate(polarity_names):
    ax = cast(Axes, axes[panel])
    tracks = eddy_tracks.loc[eddy_tracks['polarity'] == polarity]
    total = tracks.groupby('movement').size().reindex(classes, fill_value=0)
    target = tracks.loc[tracks['is_target']].groupby('movement').size().reindex(classes, fill_value=0)
    positions = np.arange(len(classes))
    all_bars = ax.bar(positions - 0.2, total.to_numpy(), width=0.38, color='#aeb5bc', label='All tracked eddies')
    target_bars = ax.bar(positions + 0.2, target.to_numpy(), width=0.38, color=polarity_colors[polarity], label='Target eddies')
    ax.bar_label(all_bars, padding=3, fontsize=10)
    ax.bar_label(target_bars, labels=[str(value) if value else '' for value in target.to_numpy()], padding=3, fontsize=10)
    ax.xaxis.set_ticks(positions)
    ax.xaxis.set_ticklabels([label[0] + ' → ' + label[1] for label in classes])
    ax.set_xlabel('First side → last side of the daily Gulf Stream axis')
    ax.set_title(f'{polarity.capitalize()}s: {len(tracks)} tracks, {int(target.sum())} target eddies')
    ax.set_ylim(0, max(int(eddy_tracks.groupby(['polarity', 'movement']).size().max()) * 1.2, 1))
    ax.grid(axis='y', alpha=0.2)
    ax.set_axisbelow(True)
    ax.legend(loc='upper left', frameon=False, fontsize=9)
    for label in classes:
        count_rows.append({'polarity': polarity, 'class': label, 'tracks': int(total[label]), 'target': int(target[label])})
axes[0].set_ylabel('Number of independent eddies')
fig.suptitle(f'Endpoint classes and the target eddies\nTarget eddies: crossed the axis, or were born within {NEAR_AXIS_KM} km of it and ended south (cyclones) or north (anticyclones)', fontsize=14)
class_counts = pd.DataFrame(count_rows)
plt.show()
print(f'Physical detections: {movement["birth_date"].min():%Y-%m-%d} to {movement["death_date"].max():%Y-%m-%d}')
print(f'CHL composite midpoint dates: {chl["date"].min():%Y-%m-%d} to {chl["date"].max():%Y-%m-%d}')
print(f'Copernicus CHL: {len(chl)} eddy-composite means from the gold table.')
print(f'Target eddies with at least one CHL composite: {int((eddy_tracks["is_target"] & eddy_tracks["n_chl_dates"].gt(0)).sum())} of {int(eddy_tracks["is_target"].sum())}.')
display(class_counts)
target_rules = {
    'crossed the axis': eddy_tracks['crossed_axis'],
    f'born within {NEAR_AXIS_KM} km, no crossing': eddy_tracks['near_axis_birth'] & ~eddy_tracks['crossed_axis'],
    'target eddies': eddy_tracks['is_target'],
}
display(pd.DataFrame([
    {'polarity': polarity, 'rule': rule, 'tracks': int((members & eddy_tracks['polarity'].eq(polarity)).sum())}
    for polarity in polarity_names for rule, members in target_rules.items()
]))

The map below draws the longest axis-crossing target eddy track of each polarity through its daily detected centers, blue for the cyclone and red for the anticyclone, with the first and last detection labeled.

A line breaks where the tracker found no eddy on one or more consecutive days. On such a day the identification step found no eddy that the tracker could link to the track, usually because no closed SSH contour near the previous position met the eddy criteria. The tracker bridges up to `eddy_track.virtual` (5) consecutive missed days with virtual positions and keeps the track as one eddy; a longer absence ends the track. The map draws only real detections, so each white gap is a run of 1 to 5 days with no detection. A gap does not remove an eddy from the age bins, because each CHL composite uses the observed boundary nearest its midpoint.


In [ ]:
# Show the longest axis-crossing target eddy track of each polarity within the full study region.
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.geoaxes import GeoAxes
from gulf_stream import load_track_observations

track_observations = load_track_observations(
    DATA_DIR / 'silver/eddy_track/cyclone',
    DATA_DIR / 'silver/eddy_track/anticyclone',
)
crossed = cast(pd.DataFrame, eddy_tracks.loc[eddy_tracks['crossed_axis']]).copy()
crossed['lifetime_days'] = (crossed['death_date'] - crossed['birth_date']).dt.days
representatives = crossed.sort_values('lifetime_days', ascending=False).drop_duplicates('polarity').set_index('polarity')

map_fig, map_ax = plt.subplots(figsize=(10, 6.4), subplot_kw={'projection': ccrs.PlateCarree()}, layout='constrained')
map_ax = cast(GeoAxes, map_ax)
map_ax.set_extent([*cfg['base']['region']['lon_range'], *cfg['base']['region']['lat_range']], crs=ccrs.PlateCarree())
map_ax.add_feature(cfeature.LAND.with_scale('50m'), facecolor='#e8e8e8', zorder=0)
map_ax.coastlines(resolution='50m', color='#555555', linewidth=0.7)
grid = map_ax.gridlines(draw_labels=True, linewidth=0.5, alpha=0.4, xlocs=range(-80, -55, 5), ylocs=range(30, 45, 2))
grid.top_labels = False
grid.right_labels = False
for polarity in polarity_names:
    track_id = int(representatives.loc[polarity, 'track_id'])
    track = cast(pd.DataFrame, track_observations.loc[
        track_observations['polarity'].eq(polarity) & track_observations['track_id'].eq(track_id)
    ]).sort_values('date')
    first, last = track.iloc[0], track.iloc[-1]
    color = polarity_colors[polarity]
    label = f'{target_labels[polarity]}: track {track_id}, {first["date"]:%Y-%m-%d} to {last["date"]:%Y-%m-%d}'
    gap_groups = track['date'].diff().dt.days.gt(1).cumsum()
    for _, segment in track.groupby(gap_groups):
        map_ax.plot(segment['center_lon'], segment['center_lat'], color=color, linewidth=2, transform=ccrs.PlateCarree(), label=label)
        label = None
    map_ax.scatter(first['center_lon'], first['center_lat'], s=85, marker='o', facecolors='white', edgecolors=color, linewidths=2, zorder=5, transform=ccrs.PlateCarree())
    map_ax.scatter(last['center_lon'], last['center_lat'], s=100, marker='^', color=color, zorder=5, transform=ccrs.PlateCarree())
    label_box = {'boxstyle': 'round,pad=0.15', 'facecolor': 'white', 'edgecolor': 'none', 'alpha': 0.85}
    map_ax.annotate('start', (first['center_lon'], first['center_lat']), xytext=(7, 7), textcoords='offset points', color=color, fontsize=10, bbox=label_box)
    map_ax.annotate('end', (last['center_lon'], last['center_lat']), xytext=(7, 7), textcoords='offset points', color=color, fontsize=10, bbox=label_box)
map_ax.legend(loc='lower right', fontsize=9, framealpha=0.95)
map_ax.set_title('Longest axis-crossing target eddy track of each polarity, detected centers only', fontsize=13)
plt.show()

In [ ]:
# Compare CHL and its change from each eddy's first available CHL observation across equal observed-track age bins; show means in which each eddy counts once per bin however many composites it has there, and bootstrap intervals that resample eddies rather than composites.
# A different set of eddies can contribute at each age. The change baseline is the first optical observation, not formation, and missing bins remain empty.
bin_edges = np.linspace(0, 1, N_AGE_BINS + 1)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
analysis = target_chl.sort_values(identity_columns + ['date']).copy()
analysis['age_bin'] = np.minimum(
    (analysis['age_frac'] * N_AGE_BINS).astype(int), N_AGE_BINS - 1,
)
analysis['initial_chl'] = analysis.groupby(identity_columns)['CHL'].transform('first')
analysis['chl_change'] = analysis['CHL'] - analysis['initial_chl']
eddy_bins = cast(pd.DataFrame, analysis.groupby(identity_columns + ['age_bin']).agg(
    chl_mean=('CHL', 'mean'), chl_change=('chl_change', 'mean'),
    n_composites=('date', 'size'),
)).reset_index()
rng = np.random.default_rng(RANDOM_SEED)
summary_rows = []
for polarity in polarity_names:
    polarity_bins = eddy_bins.loc[eddy_bins['polarity'].eq(polarity)]
    eddy_ids = sorted(polarity_bins['track_id'].unique())
    draws = rng.integers(0, len(eddy_ids), size=(N_BOOTSTRAP, len(eddy_ids)))
    for metric in ('chl_mean', 'chl_change'):
        matrix = polarity_bins.pivot(index='track_id', columns='age_bin', values=metric).reindex(index=eddy_ids, columns=range(N_AGE_BINS)).to_numpy(dtype=float)
        finite = np.isfinite(matrix)
        counts = finite.sum(axis=0)
        means = np.divide(np.nansum(matrix, axis=0), counts, out=np.full(N_AGE_BINS, np.nan), where=counts > 0)
        sampled = matrix[draws]  # (n_eddies, n_bins) -> (n_bootstrap, n_eddies, n_bins)
        sampled_counts = np.isfinite(sampled).sum(axis=1)
        sampled_means = np.divide(
            np.nansum(sampled, axis=1), sampled_counts,
            out=np.full((N_BOOTSTRAP, N_AGE_BINS), np.nan), where=sampled_counts > 0,
        )
        for age_bin in range(N_AGE_BINS):
            bootstrap_values = sampled_means[:, age_bin]  # (n_bootstrap, n_bins) -> (n_bootstrap,)
            bootstrap_values = bootstrap_values[np.isfinite(bootstrap_values)]
            low, high = (np.quantile(bootstrap_values, [0.025, 0.975]) if counts[age_bin] >= 3 else (np.nan, np.nan))
            summary_rows.append({
                'polarity': polarity, 'metric': metric, 'age_bin': age_bin,
                'age_midpoint': bin_centers[age_bin], 'mean': means[age_bin],
                'ci_low': low, 'ci_high': high, 'n_eddies': int(counts[age_bin]),
            })
lifetime_summary = pd.DataFrame(summary_rows)

life_fig, life_axes = plt.subplots(1, 2, figsize=(11, 5.2), layout='constrained')
for panel, metric in enumerate(('chl_mean', 'chl_change')):
    ax = cast(Axes, life_axes[panel])
    for polarity in polarity_names:
        result = lifetime_summary.loc[
            lifetime_summary['polarity'].eq(polarity) & lifetime_summary['metric'].eq(metric)
        ].sort_values('age_bin')
        color = polarity_colors[polarity]
        ax.plot(result['age_midpoint'], result['mean'], '-o', color=color, label=target_labels[polarity], linewidth=2)
        intervals = result.loc[result['ci_low'].notna()]
        ax.errorbar(
            intervals['age_midpoint'], intervals['mean'],
            yerr=[intervals['mean'] - intervals['ci_low'], intervals['ci_high'] - intervals['mean']],
            fmt='none', ecolor=color, capsize=4, linewidth=1.1, alpha=0.7,
        )
    ax.set_xlabel('Fraction of observed physical track')
    ax.xaxis.set_ticks(np.linspace(0, 1, 6))
    ax.set_xlim(0, 1)
    ax.grid(alpha=0.2)
life_fig.legend(*life_axes[0].get_legend_handles_labels(), loc='outside lower center', ncol=2, frameon=False, fontsize=10)
life_axes[0].set_title('Interior Copernicus CHL')
life_axes[0].set_ylabel('Mean CHL, one value per eddy (mg/m³)')
life_axes[1].set_title('Change from first CHL observation')
life_axes[1].set_ylabel('Mean ΔCHL, one value per eddy (mg/m³)')
life_axes[1].axhline(0, color='#555555', linewidth=0.8, linestyle='--')
life_fig.suptitle('Chlorophyll across observed track duration\nEach eddy counts once per bin. Error bars: 95% interval from resampling eddies', fontsize=14)
plt.show()
target_summary = cast(pd.DataFrame, analysis.groupby('polarity').agg(
    eddies=('track_id', 'nunique'), composites=('date', 'size'),
    first_composite=('date', 'min'), last_composite=('date', 'max'),
))
record_edge_counts = analysis.loc[analysis['at_record_edge']].drop_duplicates(identity_columns).groupby('polarity').size()
target_summary['tracks_at_record_edge'] = record_edge_counts.reindex(target_summary.index, fill_value=0)
clipped_rows = ((analysis['date'] < analysis['birth_date']) | (analysis['date'] > analysis['death_date'])).sum()
print(f'Exclude tracks at the physical record limits: {EXCLUDE_RECORD_EDGE_TRACKS}')
print(f'Target composite dates clipped to a track endpoint: {clipped_rows}')
print('Within a bin, the composites of one eddy are averaged into a single value first, so each eddy counts once no matter how many composites it has in the bin. No missing bin is interpolated. Error bars appear only where at least 3 eddies contribute.')
display(target_summary)
display(lifetime_summary.loc[lifetime_summary['metric'].eq('chl_mean')].drop(columns='metric').round(4))